# Stage 05 — Model Evaluation

This notebook evaluates the saved production model on the held-out test data, writes `scores.json`, and inspects sample predictions. Run Stages 01–04 first.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "research":
    os.chdir("..")
PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

In [ ]:
import json

import joblib
import numpy as np
import pandas as pd

from mlproject.components.model_evaluation import ModelEvaluation
from mlproject.config.configuration import ConfigurationManager

In [ ]:
configuration = ConfigurationManager()
evaluation_config = configuration.get_model_evaluation_config()
evaluation_config

In [ ]:
evaluator = ModelEvaluation(config=evaluation_config)
scores = evaluator.evaluate_model()
scores

In [ ]:
with evaluation_config.metric_file_name.open(encoding="utf-8") as score_file:
    saved_scores = json.load(score_file)
saved_scores

In [ ]:
model = joblib.load(evaluation_config.path_of_model)
test_data = pd.read_csv(evaluation_config.test_data_path)
features = test_data.drop(columns=[evaluation_config.target_column])
actual = test_data[evaluation_config.target_column]
raw_predictions = np.asarray(model.predict(features.head(10)))
predictions = np.clip(
    np.rint(raw_predictions),
    actual.min(),
    actual.max(),
).astype(int)
pd.DataFrame({
    "actual_quality": actual.head(10).to_numpy(),
    "predicted_quality": predictions,
})